In [51]:
import httpx
import pprint as pp
import base64
import json
from pprint import pprint

PORT = "8088"
API_URL = "http://127.0.0.1:" + PORT
DEV_TOKEN = "rl_live_8f2c1d94e6b74a03"


def auth_to_API(client: httpx.Client):
    resp = client.post(API_URL + "/v1/auth/token",
                   headers={"Authorization": f"Bearer {DEV_TOKEN}"})
    live_token = resp.json()["access_token"]
    client.headers["Authorization"] = f"Bearer {live_token}"

def b64decode_relaxed(s):
    s = s.strip()
    s += "=" * (-len(s) % 4)
    return base64.b64decode(s)

def decode_cursor(cursor: str | None) -> dict | None:
    decoded = json.loads(b64decode_relaxed(cursor))
    dict_cursor = {
        "entity": decoded["t"],
        "as_of": decoded["a"],
        "total_count": decoded["n"],
        "rows_served": decoded["r"],
        "since": decoded["s"],
        "until": decoded["u"],
        "last_seen_value": decoded["v"],
        "prev_rows_served": decoded.get("pr"),
        "prev_last_seen_value": decoded.get("pv")
    } if cursor else None
    return dict_cursor

def byte_decode(cursor) -> dict | None:
    return json.loads(b64decode_relaxed(cursor))

In [52]:
client = httpx.Client(base_url=API_URL, timeout=10,
                      mounts={"all://localhost": None,
                              "all://127.0.0.1": None})
auth_to_API(client)

response = client.get(API_URL + "/v1/orders",
                     params={"limit":50,
                             "as_of":"2026-06-01T00:00:00",
                             "since":"2026-01-01T00:00:00",
                             "until":"2026-06-01T00:00:00"})
cursor = response.json()["next_cursor"]

In [53]:
response = client.get(API_URL + "/v1/orders",
                     params={"limit":50,
                             "cursor":cursor})

In [54]:
cursor = response.json()["cursor"]
next_cursor = response.json()["next_cursor"]
pprint(decode_cursor(cursor))
pprint(decode_cursor(next_cursor))

{'as_of': '2026-05-20T15:14:14',
 'entity': 'orders',
 'last_seen_value': '2026-01-03T16:56:56',
 'prev_last_seen_value': None,
 'prev_rows_served': None,
 'rows_served': 50,
 'since': '2026-01-01T00:00:00',
 'total_count': 4628,
 'until': '2026-06-01T00:00:00'}
{'as_of': '2026-05-20T15:14:14',
 'entity': 'orders',
 'last_seen_value': '2026-01-06T08:07:42',
 'prev_last_seen_value': '2026-01-03T16:56:56',
 'prev_rows_served': 50,
 'rows_served': 100,
 'since': '2026-01-01T00:00:00',
 'total_count': 4628,
 'until': '2026-06-01T00:00:00'}


In [48]:
import sqlite3
conn = sqlite3.Connection("../landing.db")
cursor = conn.cursor()
cursor.execute("SELECT next_cursor FROM page_ledger WHERE id=1")
row = cursor.fetchone()
conn.close()
pprint(decode_cursor(row[0]))

{'as_of': '2026-05-19T17:46:27',
 'entity': 'orders',
 'last_seen_value': '2026-01-15T11:46:08',
 'prev_last_seen_value': '2026-01-15T10:42:18',
 'prev_rows_served': 326,
 'rows_served': 331,
 'since': '2026-01-15T00:00:00',
 'total_count': 305,
 'until': '2026-01-25T00:00:00'}
